# 002 — Capa de limpieza (silver): de los parquets crudos al modelo de evento unificado

Este notebook documenta y **demuestra con datos reales** la limpieza que VigIA aplica a las
fuentes de la Policía Nacional, partiendo de los parquets ya descargados en `data/bronze/`
(capa *bronze* de la arquitectura *medallion*: crudo + linaje).

**Principios de este notebook:**

1. **No duplica lógica.** Cada paso importa y ejecuta las funciones REALES de producción
   ([`ml/vigia/etl/silver.py`](../ml/vigia/etl/silver.py)); lo que se ve aquí es exactamente
   lo que ejecuta `vigia clean` dentro del contenedor.
2. **Solo lectura.** No escribe `data/silver/`, ni `reports/` (eso lo hace el pipeline:
   `make docker-pipeline`). Al final se **verifica** que el resultado construido en memoria
   coincide con el artefacto persistido y con el reporte de calidad versionado.

**El reto técnico central** (detalle en [docs/DATA_DICTIONARY.md](../docs/DATA_DICTIONARY.md)):
las 16 fuentes de eventos vienen en **dos familias de esquema** y **dos formatos de fecha**:

| | Familia A | Familia B |
|---|---|---|
| Código de municipio | `cod_muni` (5 dígitos DANE) | `codigo_dane` (8 dígitos → 5) |
| Fecha | ISO (`2003-01-01T00:00:00`) | `dd/mm/yyyy` |
| Ejemplos | homicidios, hurto a personas, extorsión… | violencia intrafamiliar, amenazas, capturas… |

Además: los **nombres** de municipio/departamento de las fuentes son inconsistentes (se
reemplazan por los oficiales **DIVIPOLA/DANE**), y cada categoría se clasifica por
**naturaleza** (*delito* vs *respuesta institucional*) para no confundir incidencia con
actividad operativa.

> Requisito: el paquete instalado en modo editable (`cd ml && pip install -e ".[dev]"`) y el
> bronze ya ingerido (`vigia ingest` o `make docker-pipeline`).

In [1]:
import sys
from pathlib import Path

# El paquete se instala con `cd ml && pip install -e ".[dev]"`; si no está instalado,
# se usa directamente desde el repositorio (el notebook vive en notebooks/).
try:
    import vigia  # noqa: F401
except ModuleNotFoundError:
    for base in (Path.cwd(), Path.cwd().parent):
        if (base / "ml" / "vigia").exists():
            sys.path.insert(0, str(base / "ml"))
            break
import json

import pandas as pd
import pyarrow.parquet as pq

from vigia.config import settings
from vigia.datasets import CATALOG, EVENT_DATASETS
from vigia.etl import silver
from vigia.etl.quality import quality_report

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

BRONZE = settings.bronze_dir
print("Python", sys.version.split()[0], "· pandas", pd.__version__)
print("Lago de datos:", settings.data_dir)

Python 3.13.9 · pandas 2.3.3
Lago de datos: D:\GitHub\vigia\data


## 1. Inventario: el catálogo declarativo y los parquets bronze

El catálogo (`ml/vigia/datasets.py`) es **declarativo**: cada fuente lleva su familia de
esquema, formato de fecha, categoría por defecto y naturaleza. Añadir una fuente de la
familia A es *incorporación directa*: una entrada en `CATALOG`, sin tocar `silver.py`.

In [2]:
inventario = []
for spec in CATALOG:
    src = BRONZE / f"{spec.id}.parquet"
    inventario.append(
        {
            "fuente": spec.id,
            "soda_id": spec.soda_id,
            "familia": spec.schema_family,
            "formato_fecha": spec.date_format,
            "categoria_por_defecto": spec.categoria,
            "naturaleza": spec.naturaleza,
            "filas_bronze": pq.read_metadata(src).num_rows if src.exists() else None,
        }
    )
inventario = pd.DataFrame(inventario)
total = inventario["filas_bronze"].sum()
print(f"{len(inventario)} fuentes de eventos · {total:,.0f} filas crudas en bronze".replace(",", "."))
inventario

16 fuentes de eventos · 8.810.716 filas crudas en bronze


,fuente,soda_id,familia,formato_fecha,categoria_por_defecto,naturaleza,filas_bronze
0,homicidios,m8fd-ahd9,A,iso,HOMICIDIO,delito,339653
1,hurto_vehiculos,csb4-y6v2,A,iso,HURTO_VEHICULOS,delito,382563
2,violencia_intrafamiliar,vuyt-mqpw,B,dmy,VIOLENCIA_INTRAFAMILIAR,delito,682558
3,amenazas,meew-mguv,B,dmy,AMENAZAS,delito,650347
4,reporte_capturas,3jdh-nmwu,B,dmy,CAPTURAS,respuesta,3673572
5,incautacion_armas,2iz5-9bbz,B,dmy,INCAUTACION_ARMAS,respuesta,421224
6,recuperacion_vehiculos,dhy3-732k,B,dmy,RECUPERACION_VEHICULOS,respuesta,276743
7,hurto_modalidades,d4fr-sbn2,B,dmy,HURTO_OTRAS_MODALIDADES,delito,44169
8,hurto_personas,4rxi-8m8d,A,iso,HURTO_PERSONAS,delito,641724
9,hurto_residencias,7mn7-vzqp,A,iso,HURTO_RESIDENCIAS,delito,609597


## 2. Las dos familias de esquema, vistas en el dato crudo

Se cargan una fuente de cada familia y se comparan las columnas que la capa silver debe
reconciliar: el código territorial y la fecha del hecho.

In [3]:
fam_a = pd.read_parquet(BRONZE / "homicidios.parquet")
fam_b = pd.read_parquet(BRONZE / "violencia_intrafamiliar.parquet")

cols_a = [c for c in ("fecha_hecho", "cod_depto", "cod_muni", "departamento", "municipio", "cantidad") if c in fam_a.columns]
cols_b = [c for c in ("fecha_hecho", "codigo_dane", "departamento", "municipio", "cantidad") if c in fam_b.columns]

print("Familia A — homicidios:", sorted(fam_a.columns.tolist()))
display(fam_a[cols_a].head(3))
print("Familia B — violencia_intrafamiliar:", sorted(fam_b.columns.tolist()))
display(fam_b[cols_b].head(3))

Familia A — homicidios: ['_modalidad_presunta', 'arma_medio', 'cantidad', 'cod_depto', 'cod_muni', 'departamento', 'fecha_hecho', 'municipio', 'sexo', 'spoa_caracterizacion', 'zona']


,fecha_hecho,cod_depto,cod_muni,departamento,municipio,cantidad
0,2003-01-01T00:00:00.000,11,11001,BOGOTA D.C.,BOGOTA D.C.,1
1,2003-01-01T00:00:00.000,11,11001,BOGOTA D.C.,BOGOTA D.C.,1
2,2003-01-01T00:00:00.000,11,11001,BOGOTA D.C.,BOGOTA D.C.,1


Familia B — violencia_intrafamiliar: ['armas_medios', 'cantidad', 'codigo_dane', 'departamento', 'fecha_hecho', 'genero', 'grupo_etario', 'municipio']


,fecha_hecho,codigo_dane,departamento,municipio,cantidad
0,10/03/2026,17001000,CALDAS,Manizales (CT),2
1,15/03/2026,08433000,ATLÁNTICO,Malambo,1
2,09/03/2026,08421000,ATLÁNTICO,Luruaco,1


## 3. Paso 1 — Fechas: dos formatos, una sola salida (`_parse_dates`)

La familia A trae fechas ISO; la B, `dd/mm/yyyy`. `silver._parse_dates` interpreta ambas y
garantiza una salida uniforme `datetime64[ns]` **sin zona horaria** (a escala real, la
interpretación mixta puede inferir zona en algunos registros y romper comparaciones). Las
fechas no interpretables quedan nulas y la fila se descarta después.

In [4]:
demo_a = fam_a["fecha_hecho"].astype("string").dropna().iloc[:4]
demo_b = fam_b["fecha_hecho"].astype("string").dropna().iloc[:4]

display(pd.DataFrame({
    "crudo (familia A, ISO)": demo_a.to_numpy(),
    "interpretado": silver._parse_dates(demo_a, "iso").dt.date.to_numpy(),
}))
display(pd.DataFrame({
    "crudo (familia B, dd/mm/yyyy)": demo_b.to_numpy(),
    "interpretado": silver._parse_dates(demo_b, "dmy").dt.date.to_numpy(),
}))

no_parseables = silver._parse_dates(fam_b["fecha_hecho"], "dmy").isna().sum()
print(f"Fechas no interpretables en violencia_intrafamiliar: {no_parseables:,}".replace(",", "."))

,"crudo (familia A, ISO)",interpretado
0,2003-01-01T00:00:00.000,2003-01-01
1,2003-01-01T00:00:00.000,2003-01-01
2,2003-01-01T00:00:00.000,2003-01-01
3,2003-01-01T00:00:00.000,2003-01-01


,"crudo (familia B, dd/mm/yyyy)",interpretado
0,10/03/2026,2026-03-10
1,15/03/2026,2026-03-15
2,09/03/2026,2026-03-09
3,14/03/2026,2026-03-14


Fechas no interpretables en violencia_intrafamiliar: 0


## 4. Paso 2 — Código DANE de municipio: 8 dígitos → 5 (`_to_dane5`)

La familia B trae `codigo_dane` de **8 dígitos** (municipio + 3 del centro poblado), a veces
como número con decimal. Se recorta a los 5 primeros y se valida: un código DANE real nunca
tiene `00` como departamento (los departamentos van de 05 a 99), así que los marcadores
`000…` de "código en blanco" quedan nulos y la fila se descarta — sin esto, esos códigos
inflaban el conteo de departamentos del tablero.

In [5]:
demo = fam_b["codigo_dane"].astype("string").dropna().iloc[:4]
display(pd.DataFrame({
    "codigo_dane crudo (familia B)": demo.to_numpy(),
    "cod_municipio (5 dígitos DANE)": silver._to_dane5(demo, "B").to_numpy(),
}))

codigos = silver._to_dane5(fam_b["codigo_dane"], "B")
descartados = codigos.isna().sum()
print(f"Códigos inválidos (en blanco o marcador '00…') que se descartan: "
      f"{descartados:,} de {len(codigos):,}".replace(",", "."))

,codigo_dane crudo (familia B),cod_municipio (5 dígitos DANE)
0,17001000,17001
1,08433000,08433
2,08421000,08421
3,41551000,41551


Códigos inválidos (en blanco o marcador '00…') que se descartan: 1 de 682.558


## 5. Paso 3 — Texto: acentos, sufijos `(CT)` y espacios (`_norm_text`)

Distintas fuentes escriben el mismo territorio con o sin tilde, con el sufijo `(CT)` de
capital, o con espacios dobles — sin normalizar, un mismo municipio se duplicaría. La
normalización pasa a mayúsculas, elimina diacríticos (NFKD), quita paréntesis y colapsa
espacios; los vacíos se imputan con el marcador `NO REPORTADO` (su frecuencia real se mide
aparte en el reporte de calidad, sin maquillar la completitud).

In [6]:
crudos = pd.Series(sorted(set(
    fam_a["municipio"].dropna().astype(str).tolist()[:40000]
    + fam_b["departamento"].dropna().astype(str).tolist()[:40000]
)))
interesantes = crudos[crudos.str.contains(r"\(CT\)|[ÁÉÍÓÚÑáéíóúñ]", regex=True)].iloc[:8]
pd.DataFrame({
    "crudo": interesantes.to_numpy(),
    "normalizado": silver._norm_text(interesantes).to_numpy(),
})

,crudo,normalizado
0,ATLÁNTICO,ATLANTICO
1,BOLÍVAR,BOLIVAR
2,BOYACÁ,BOYACA
3,BRICEÑO,BRICENO
4,CAQUETÁ,CAQUETA
5,CAÑASGORDAS,CANASGORDAS
6,CHOCÓ,CHOCO
7,COVEÑAS,COVENAS


## 6. Paso 4 — Categoría: vocabulario controlado y separador canónico

La mayoría de fuentes aporta **una** categoría (la del catálogo). Tres traen la categoría en
una columna de texto (`tipo_delito` / `tipo_de_hurto`) con **vocabulario controlado**
(verificado contra la API — no hace falta diccionario de sinónimos): se les quita el prefijo
legal `ARTICULO n.` y se unifica el separador a guion bajo (`HURTO MOTOCICLETAS` →
`HURTO_MOTOCICLETAS`), la misma convención de las categorías por defecto.

In [7]:
for nombre, archivo, col in [
    ("hurto_vehiculos", "hurto_vehiculos.parquet", "tipo_delito"),
    ("hurto_modalidades", "hurto_modalidades.parquet", "tipo_de_hurto"),
    ("secuestro", "secuestro.parquet", "tipo_delito"),
]:
    s = pd.read_parquet(BRONZE / archivo, columns=[col])[col]
    print(f"{nombre} · {col}: {sorted(s.dropna().unique().tolist())}")

hurto_vehiculos · tipo_delito: ['ARTICULO 239. HURTO AUTOMOTORES', 'ARTICULO 239. HURTO MOTOCICLETAS']
hurto_modalidades · tipo_de_hurto: ['HURTO ABIGEATO', 'HURTO ENTIDADES FINANCIERAS', 'HURTO PIRATERÍA TERRESTRE']
secuestro · tipo_delito: ['ARTICULO 168. SECUESTRO SIMPLE', 'ARTICULO 169. SECUESTRO EXTORSIVO']


## 7. `normalize()`: todos los pasos juntos sobre una fuente

`silver.normalize(df, spec)` aplica los pasos anteriores y descarta las filas **no
localizables** (sin fecha válida o sin código de municipio) y las anómalas (año < 1990,
cantidad ≤ 0). El resultado es el **esquema unificado de evento** (15 columnas), idéntico
para las 16 fuentes.

In [8]:
spec_b = EVENT_DATASETS["violencia_intrafamiliar"]
norm_b = silver.normalize(fam_b, spec_b)
print(f"{len(fam_b):,} filas crudas → {len(norm_b):,} eventos válidos "
      f"({len(fam_b) - len(norm_b):,} descartadas)".replace(",", "."))
norm_b.head(4)

682.558 filas crudas → 682.557 eventos válidos (1 descartadas)


,fecha,anio,mes,cod_departamento,departamento,cod_municipio,municipio,zona,categoria,arma_medio,sexo,grupo_etario,cantidad,fuente,ingested_at
0,2026-03-10,2026,3,17,CALDAS,17001,MANIZALES,NO REPORTADO,VIOLENCIA_INTRAFAMILIAR,SIN EMPLEO DE ARMAS,FEMENINO,ADULTOS,2,violencia_intrafamiliar,2026-07-07 03:13:58.587344+00:00
1,2026-03-15,2026,3,08,ATLANTICO,08433,MALAMBO,NO REPORTADO,VIOLENCIA_INTRAFAMILIAR,SIN EMPLEO DE ARMAS,FEMENINO,ADULTOS,1,violencia_intrafamiliar,2026-07-07 03:13:58.587344+00:00
2,2026-03-09,2026,3,08,ATLANTICO,08421,LURUACO,NO REPORTADO,VIOLENCIA_INTRAFAMILIAR,SIN EMPLEO DE ARMAS,FEMENINO,ADULTOS,1,violencia_intrafamiliar,2026-07-07 03:13:58.587344+00:00
3,2026-03-14,2026,3,41,HUILA,41551,PITALITO,NO REPORTADO,VIOLENCIA_INTRAFAMILIAR,SIN EMPLEO DE ARMAS,FEMENINO,ADULTOS,1,violencia_intrafamiliar,2026-07-07 03:13:58.587344+00:00


## 8. Normalización de las 16 fuentes (el mismo bucle de `build_silver`, en memoria)

Se repite el bucle exacto de producción — `normalize()` por fuente y concatenación — pero
**sin persistir**. La tabla muestra cuántas filas crudas aporta cada fuente y cuántas
sobreviven a la limpieza.

In [9]:
frames, resumen = [], []
for spec in CATALOG:
    src = BRONZE / f"{spec.id}.parquet"
    if not src.exists():
        print(f"AVISO: bronze ausente para {spec.id}; omitido")
        continue
    raw = pd.read_parquet(src)
    norm = silver.normalize(raw, spec)
    frames.append(norm)
    resumen.append({
        "fuente": spec.id,
        "filas_bronze": len(raw),
        "eventos_validos": len(norm),
        "descartadas": len(raw) - len(norm),
        "descarte_pct": round(100 * (1 - len(norm) / len(raw)), 2) if len(raw) else None,
    })
eventos = pd.concat(frames, ignore_index=True)
del frames, raw, norm

resumen = pd.DataFrame(resumen).sort_values("filas_bronze", ascending=False).reset_index(drop=True)
print(f"Total unificado: {len(eventos):,} eventos".replace(",", "."))
resumen

Total unificado: 8.810.230 eventos


,fuente,filas_bronze,eventos_validos,descartadas,descarte_pct
0,reporte_capturas,3673572,3673466,106,0.00
1,violencia_intrafamiliar,682558,682557,1,0.00
2,amenazas,650347,650347,0,0.00
3,hurto_personas,641724,641359,365,0.06
4,hurto_residencias,609597,609597,0,0.00
5,delitos_informaticos,491867,491867,0,0.00
6,delitos_sexuales,438526,438526,0,0.00
7,incautacion_armas,421224,421215,9,0.00
8,hurto_vehiculos,382563,382563,0,0.00
9,homicidios,339653,339653,0,0.00


## 9. Nombres oficiales DANE (DIVIPOLA) y deduplicación

Los nombres de las fuentes delictivas **no se usan como identidad**: la clave es el código
DANE, y el nombre se reemplaza por el oficial de **DIVIPOLA** (donde DIVIPOLA no tiene el
código, se conserva el normalizado de la fuente). Después se eliminan los duplicados exactos
— el último paso de `build_silver`.

In [10]:
antes = sorted(eventos.loc[eventos["cod_municipio"] == "11001", "municipio"].unique().tolist())
eventos = silver._apply_official_names(eventos)
despues = sorted(eventos.loc[eventos["cod_municipio"] == "11001", "municipio"].unique().tolist())
print("Nombres para el código 11001 ANTES del cruce :", antes)
print("Nombres para el código 11001 DESPUÉS del cruce:", despues)

n0 = len(eventos)
eventos = eventos.drop_duplicates()
print(f"Duplicados exactos eliminados: {n0 - len(eventos):,}".replace(",", "."))

[22:21:27] INFO     DIVIPOLA: 1122 municipios oficiales cargados

[22:21:37] INFO     Nombres oficiales DIVIPOLA aplicados a 100.0% de los eventos

Nombres para el código 11001 ANTES del cruce : ['BOGOTA D.C.']
Nombres para el código 11001 DESPUÉS del cruce: ['BOGOTÁ, D.C.']


Duplicados exactos eliminados: 3.769.661


## 10. Naturaleza: delito vs respuesta institucional

Una fracción relevante de los hechos registrados es **respuesta institucional** (capturas,
incautaciones, recuperaciones), no delito. Confundirlas inflaría la "incidencia" del
tablero, y una alerta de anomalías sobre un repunte de capturas sería un contrasentido
(es buena noticia). La clasificación vive en una sola fuente de verdad:
`datasets.naturaleza()` / `RESPONSE_CATEGORIES`.

In [11]:
from vigia.datasets import naturaleza

por_naturaleza = (
    eventos.assign(naturaleza=eventos["categoria"].map(naturaleza))
    .groupby("naturaleza")["cantidad"]
    .agg(hechos="sum")
)
por_naturaleza["participacion_pct"] = (100 * por_naturaleza["hechos"] / por_naturaleza["hechos"].sum()).round(2)
display(por_naturaleza)

print("Categorías clasificadas como respuesta:", sorted(
    eventos.loc[eventos["categoria"].map(naturaleza) == "respuesta", "categoria"].unique().tolist()
))

,hechos,participacion_pct
naturaleza,,
delito,7558891,81.85
respuesta,1676233,18.15


Categorías clasificadas como respuesta: ['CAPTURAS', 'INCAUTACION_ARMAS', 'RECUPERACION_VEHICULOS']


## 11. Verificación: lo construido aquí coincide con los artefactos de producción

El silver construido en memoria se contrasta contra (a) el parquet persistido por el último
`vigia clean` y (b) el reporte de calidad versionado
([reports/silver_quality.json](../reports/silver_quality.json)). Si se reingirió bronze
después de la última ejecución del pipeline, pueden aparecer diferencias legítimas
(el portal publica actualizaciones mensuales).

In [12]:
qrep = json.loads((settings.reports_dir / "silver_quality.json").read_text(encoding="utf-8"))
persistido_filas = pq.read_metadata(settings.silver_dir / "eventos.parquet").num_rows

checks = [
    ("filas del evento unificado", len(eventos), qrep["filas"]),
    ("filas del parquet silver persistido", len(eventos), persistido_filas),
    ("total de hechos (suma de cantidad)", int(eventos["cantidad"].sum()), qrep["total_hechos"]),
    ("municipios únicos (código DANE)", int(eventos["cod_municipio"].nunique()), qrep["municipios_unicos"]),
    ("fecha mínima", str(eventos["fecha"].min().date()), qrep["rango_fechas"]["min"]),
    ("fecha máxima", str(eventos["fecha"].max().date()), qrep["rango_fechas"]["max"]),
]
verif = pd.DataFrame(
    [{"medida": m, "este notebook": a, "producción (versionado)": b,
      "coincide": "✔" if a == b else "✖"} for m, a, b in checks]
)
display(verif)
assert (verif["coincide"] == "✔").all() or True  # informe visual; no aborta si el bronze cambió

,medida,este notebook,producción (versionado),coincide
0,filas del evento unificado,5040569,5040569,✔
1,filas del parquet silver persistido,5040569,5040569,✔
2,total de hechos (suma de cantidad),9235124,9235124,✔
3,municipios únicos (código DANE),1126,1126,✔
4,fecha mínima,2003-01-01,2003-01-01,✔
5,fecha máxima,2026-05-31,2026-05-31,✔


In [13]:
# Conteo por fuente: notebook vs reporte versionado (deben coincidir uno a uno).
conteo_nb = eventos["fuente"].value_counts()
comp_fuentes = pd.DataFrame({
    "este notebook": conteo_nb,
    "silver_quality.json": pd.Series(qrep["fuentes"]),
})
comp_fuentes["coincide"] = (comp_fuentes["este notebook"] == comp_fuentes["silver_quality.json"]).map({True: "✔", False: "✖"})
comp_fuentes.sort_values("este notebook", ascending=False)

,este notebook,silver_quality.json,coincide
reporte_capturas,1237427,1237427,✔
violencia_intrafamiliar,681664,681664,✔
hurto_personas,641359,641359,✔
amenazas,428409,428409,✔
hurto_vehiculos,382563,382563,✔
hurto_residencias,333220,333220,✔
delitos_sexuales,291469,291469,✔
homicidios,267978,267978,✔
incautacion_armas,251247,251247,✔
delitos_informaticos,182538,182538,✔


In [14]:
# Y el % real de "NO REPORTADO" por campo (subregistro declarado, no maquillado):
# se recalcula con la MISMA función de producción y se compara con el versionado.
q_nb = json.loads(quality_report(eventos))
pd.DataFrame({
    "este notebook (%)": pd.Series(q_nb["placeholders_pct"]),
    "reporte versionado (%)": pd.Series(qrep["placeholders_pct"]),
}).fillna(0.0)

,este notebook (%),reporte versionado (%)
municipio,0.00,0.00
zona,81.04,81.04
arma_medio,74.25,74.25
sexo,41.65,41.65
grupo_etario,52.62,52.62


## Cierre

- La limpieza demostrada es la de producción: **mismas funciones, mismos resultados** que
  `vigia clean` (verificado arriba contra el parquet y el reporte versionados).
- En la etapa siguiente del pipeline, la capa **gold** (`ml/vigia/etl/gold.py`) agrega este
  evento unificado a la serie mensual `municipio × categoría` (calendario por serie, sin ceros
  inventados), le cruza la **población DANE** y deriva los agregados del tablero.
- El modelo de pronóstico que consume esa serie se documenta en
  [003_Modelo_Pronostico.ipynb](003_Modelo_Pronostico.ipynb).
- Este notebook **no escribió ningún artefacto**: la ejecución oficial del pipeline es
  `make docker-pipeline` (dentro de Docker).